In [1]:
from top2vec import Top2Vec
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def top2vec_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time()
    top2vec_model = Top2Vec(
        texts,
        embedding_model='all-MiniLM-L6-v2',
        speed="learn"
    )
    cluster_topics = (top2vec_model.get_topics())[0]
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)

    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    top2vec_analysis(nurse_notes[key]['Note'])
    all_texts.extend(nurse_notes[key]['Note'])

-----------P1-----------


2026-02-02 10:24:59,746 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:24:59,783 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Number of texts: 603


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-02 10:25:02,692 - top2vec - INFO - Creating joint document/word embedding
2026-02-02 10:25:08,902 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-02 10:25:49,828 - top2vec - INFO - Finding dense areas of documents
2026-02-02 10:25:49,851 - top2vec - INFO - Finding topics


...

Number of texts: 631


/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:33:42,869 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-02 10:33:48,906 - top2vec - INFO - Creating joint document/word embedding
2026-02-02 10:34:25,473 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-02 10:34:28,073 - top2vec - INFO - Finding dense areas of documents
2026-02-02 10:34:28,131 - top2vec - INFO - Finding topics


Coherence: 0.36440951228465035
Diversity: 0.475
Inverse Redundancy: 0.3833333333333333
Time (seconds): 45.785497188568115
----- Cluster Topics -----
['resident' 'meds' 'administered' 'concerns' 'medications' 'care'
 'compliant' 'settled' 'assisted' 'maintained']
['meds' 'resident' 'adls' 'compliant' 'medications' 'administered'
 'settled' 'maintained' 'safety' 'night']
['comfortable' 'bed' 'resident' 'asleep' 'toileting' 'meds' 'concerns'
 'settled' 'compliant' 'morning']
['resident' 'meds' 'compliant' 'concerns' 'care' 'administered' 'settled'
 'sensor' 'maintained' 'medications']
Number of Topics: 4


In [7]:
top2vec_analysis(all_texts)

Number of texts: 12373


2026-02-02 10:34:40,250 - top2vec - INFO - Pre-processing documents for training
/Users/isabel/anaconda3/envs/realData/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-02-02 10:34:43,510 - top2vec - INFO - Downloading all-MiniLM-L6-v2 model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-02 10:34:46,817 - top2vec - INFO - Creating joint document/word embedding
2026-02-02 10:35:57,058 - top2vec - INFO - Creating lower dimension embedding of documents
2026-02-02 10:37:04,929 - top2vec - INFO - Finding dense areas of documents
2026-02-02 10:37:07,460 - top2vec - INFO - Finding topics


Coherence: 0.34158639825397125
Diversity: 0.14432989690721648
Inverse Redundancy: 0.702061855670103
Time (seconds): 147.88338088989258
----- Cluster Topics -----
['resident' 'residents' 'resting' 'hospital' 'appointment' 'med' 'comfort'
 'relaxing' 'comfortable' 'settled']
['resident' 'residents' 'med' 'hospital' 'appointment' 'meds' 'doctor'
 'prescribed' 'visit' 'settled']
['med' 'resident' 'adls' 'adl' 'meds' 'residents' 'prescribed'
 'appointment' 'hospital' 'medication']
['resting' 'slept' 'resident' 'hospital' 'residents' 'asleep' 'sleeping'
 'tele' 'bell' 'alarm']
['resident' 'residents' 'hospital' 'med' 'appointment' 'nurse' 'assisted'
 'administered' 'resting' 'prescribed']
['meds' 'med' 'resident' 'residents' 'adls' 'hospital' 'compliant'
 'prescribed' 'appointment' 'medicines']
['resting' 'slept' 'resident' 'sleeping' 'residents' 'hospital' 'bed'
 'nurse' 'asleep' 'appointment']
['slept' 'asleep' 'sleeping' 'resting' 'sleep' 'resident' 'awake'
 'settled' 'relaxing' 'resident